# MaxCut with PCE and QAOA

This tutorial compares QASP's Pauli Correlation Encoding (PCE) and QAOA on the **same graphs**, with exact optima for validation, three initializations per graph, equal optimizer-step budgets, and results both before and after the **same single-bit improvement sweep**. It also checks PCE values and gradients across compatible simulators and draws both circuits in Matplotlib and TikZ.

Reference: M. Sciorilli et al., [*Towards large-scale quantum optimization solvers with few qubits*, Nature Communications 16, 476 (2025)](https://doi.org/10.1038/s41467-024-55346-z); [authors' code](https://github.com/MarcoSciorilli/PCE/tree/0.1).

Run this notebook from a QASP source checkout (`python -m pip install -e '.[dev,preview]'`). The notebook finds that checkout when started from `tutorials/optimization`. Optional PennyLane tensor tests require `python -m pip install quimb cotengra`. TikZ previews require pdfLaTeX/Quantikz and Ghostscript; their absence does not prevent training. No circuit files are saved automatically.

In [ ]:
from pathlib import Path
import sys
import inspect
import importlib.metadata as metadata
from itertools import combinations, product
from time import perf_counter
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

# Use this source checkout when the notebook is run from inside it.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "qasp" / "optimization" / "pce.py").is_file():
        sys.path.insert(0, str(candidate))
        break
import qasp
from qasp import QUBO, PCE, QAOA, Normal, draw_pce, draw_qaoa
assert "postprocess" in inspect.signature(PCE.solve).parameters, "Install the updated QASP source checkout."
torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})
print("QASP source:", qasp.__file__)
print({name: metadata.version(name) for name in ("torch", "pennylane", "numpy")})

## 1. Loss, encoding and fair comparison

For each original binary variable, PCE chooses a same-axis Pauli string $P_i$, measures $c_i=\langle P_i\rangle$, and forms $s_i=\tanh(\alpha c_i)$. Its default order-two XYZ encoding has capacity $3\binom{n}{2}$ on $n$ qubits. With each graph edge counted once, the paper's loss is

$$L_{\mathrm{PCE}}=\sum_{(i,j)\in E}w_{ij}s_i s_j+\beta\nu\left(\frac{1}{N}\sum_i s_i^2\right)^2,$$

where $\beta=0.5$, $\alpha=n^{\lfloor r/2\rfloor}$ by default, and for a positive weighted connected graph $\nu=w(G)/2+w(T_{\min})/4$. Disconnected graphs use the corresponding forest. The gradient includes the classical chain rule through tanh and the regularizer. The final sign-decoded bit is $b_i=(1-\operatorname{sgn}c_i)/2$; QASP maps exact zero to bit zero.

QAOA uses one qubit per variable and minimizes the expected **negative cut weight**. Its selected candidate below is the most probable computational-basis bitstring, not the best of many samples. PCE's loss is not itself a cut value and should not share a numerical comparison axis with the QAOA loss.

Both methods get Adam, the same learning rate and number of updates, and Normal initializers. Their parameter counts and ansatz structures differ, so this is an illustrative comparison, not equal computational resources or a reproduction of the paper's benchmark. We report every restart, both raw and improved solutions, and measured training times.

In [ ]:
SEEDS = [3, 7, 11]
STEPS = 60
LEARNING_RATE = 0.03

graph_specs = [
    {"name": "Cycle · 4", "n": 4,
     "edges": [(i, (i + 1) % 4, 1.0) for i in range(4)]},
    {"name": "Chorded ring · 6", "n": 6,
     "edges": [(i, (i + 1) % 6, 1.0) for i in range(6)] + [(0, 3, 1.0), (1, 4, 1.0), (2, 5, 1.0)]},
]
rng = np.random.default_rng(2026)
edges = [(i, i + 1, float(rng.uniform(0.5, 2.0))) for i in range(7)]
edges += [(i, j, float(rng.uniform(0.5, 2.0)))
          for i, j in combinations(range(8), 2) if j > i + 1 and rng.random() < 0.3]
graph_specs.append({"name": "Weighted random · 8", "n": 8, "edges": edges})

def cut_weight(bits, edges):
    return sum(w for i, j, w in edges if bits[i] != bits[j])

for graph in graph_specs:
    graph["problem"] = QUBO.maxcut(graph["n"], graph["edges"])
    # Independent discrete reference, not a quantum solver or its loss.
    candidates = ["".join(bits) for bits in product("01", repeat=graph["n"])]
    graph["exact_cut"] = max(cut_weight(bits, graph["edges"]) for bits in candidates)
    assert all(np.isclose(graph["problem"].energy(bits), -cut_weight(bits, graph["edges"])) for bits in candidates)

display(pd.DataFrame([{"Graph": g["name"], "Variables": g["n"], "Edges": len(g["edges"]),
                       "Exact maximum cut": g["exact_cut"]} for g in graph_specs]))

## 2. Configure models and optimize

PCE uses the configurable two-layer Y/Z/ZZ ansatz with initial Hadamards. QAOA uses two cost/mixer layers. Both execute on `default.qubit` with analytic Torch backpropagation. Edit the dictionaries below to explore layer counts, initializers and learning rates. The original QUBO graph always has `N` variables even though the PCE register is smaller.

The timer records one cold loss-plus-gradient warmup separately; training time then includes the optimizer, final readout and result construction. Times are local observations without repeated timing trials. Both models are warmed before training, and each receives a fresh initializer with the stated seed.

In [ ]:
def make_models(problem, seed):
    pce_config = {"backend": "torch", "device": "default.qubit", "diff_method": "backprop",
                  "correlation_order": 2, "paulis": "XYZ", "regularization": 0.5,
                  "init": Normal(0, 0.3), "seed": seed}
    qaoa_config = {"backend": "torch", "device": "default.qubit", "diff_method": "backprop",
                   "n_layers": 2, "init": Normal(0, 0.3), "seed": seed}
    return PCE(problem, **pce_config), QAOA(problem, **qaoa_config)

rows, runs = [], {}
for graph in graph_specs:
    for seed in SEEDS:
        pce, qaoa = make_models(graph["problem"], seed)
        for label, model in [("PCE", pce), ("QAOA", qaoa)]:
            theta = model.initial_parameters()
            start = perf_counter()
            if label == "PCE": model.loss(theta)
            else: model.expectation(theta)
            model.gradient(theta)
            warmup = perf_counter() - start
            start = perf_counter()
            result = model.solve(theta, optimizer="Adam",
                                 optimizer_options={"max_steps": STEPS, "lr": LEARNING_RATE})
            elapsed = perf_counter() - start
            raw = result.metadata["raw_bitstring"] if label == "PCE" else result.bitstring
            # Same original problem and same sequential sweep for both algorithms.
            improved = pce.improve_bitstring(raw)
            if label == "PCE": assert improved == result.bitstring
            raw_cut, improved_cut = cut_weight(raw, graph["edges"]), cut_weight(improved, graph["edges"])
            assert improved_cut >= raw_cut - 1e-10
            assert improved_cut <= graph["exact_cut"] + 1e-10
            rows.append({"graph": graph["name"], "algorithm": label, "seed": seed,
                         "variables": graph["n"], "qubits": model.n_qubits,
                         "parameters": len(theta), "raw_cut": raw_cut, "improved_cut": improved_cut,
                         "raw_ratio": raw_cut / graph["exact_cut"],
                         "improved_ratio": improved_cut / graph["exact_cut"],
                         "exact_cut": graph["exact_cut"], "warmup_seconds": warmup,
                         "training_seconds": elapsed, "raw_bitstring": raw, "improved_bitstring": improved})
            runs[(graph["name"], seed, label)] = (model, result)
results = pd.DataFrame(rows)
display(results.round(4))

## 3. Compare solution quality and resources

Bars show the mean approximation ratio over all three restarts; error bars show the observed range. Raw results isolate the quantum-model output. The improved results include the same classical sweep. A ratio of one means the exact small-instance optimum was reached. No result is selected by looking at the known optimum.

In [ ]:
summary = results.groupby(["graph", "algorithm"], sort=False).agg(
    qubits=("qubits", "first"), parameters=("parameters", "first"),
    raw_ratio=("raw_ratio", "mean"), improved_ratio=("improved_ratio", "mean"),
    mean_warmup_seconds=("warmup_seconds", "mean"), mean_training_seconds=("training_seconds", "mean"))
display(summary.round(4))
fig, ax = plt.subplots(figsize=(10, 4.6))
x = np.arange(len(graph_specs)); width = 0.19
for index, (algorithm, column, label, color) in enumerate([
    ("PCE", "raw_ratio", "PCE · raw", "#73b6e6"),
    ("PCE", "improved_ratio", "PCE · improved", "#1267a6"),
    ("QAOA", "raw_ratio", "QAOA · raw", "#e5b379"),
    ("QAOA", "improved_ratio", "QAOA · improved", "#a65f12"),
]):
    data = [results[(results.graph == g["name"]) & (results.algorithm == algorithm)][column].to_numpy() for g in graph_specs]
    means = np.array([v.mean() for v in data])
    errors = np.array([means - [v.min() for v in data], [v.max() for v in data] - means])
    ax.bar(x + (index - 1.5)*width, means, width, yerr=errors, capsize=3, label=label, color=color)
ax.axhline(1, color="#52636e", linestyle="--", linewidth=1)
ax.set(xticks=x, xticklabels=[g["name"] for g in graph_specs], ylabel="Cut / exact maximum cut", ylim=(0, 1.16))
ax.legend(ncol=2, loc="lower right"); fig.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for algorithm, ax in zip(["PCE", "QAOA"], axes):
    for seed in SEEDS:
        _, result = runs[(graph_specs[-1]["name"], seed, algorithm)]
        values = np.asarray(result.history)
        ax.plot(np.arange(1, len(values)+1), values if algorithm == "PCE" else -values, label=f"seed {seed}")
    ax.set(xlabel="Optimizer update", title=algorithm,
           ylabel="PCE surrogate (minimize)" if algorithm == "PCE" else "Expected cut (maximize)")
    ax.legend()
fig.suptitle("Weighted random graph · distinct training objectives")
fig.tight_layout(); plt.show()

## 4. Verify PCE outputs and gradients on different simulators

First keep the same noncommuting PCE circuit, encoding and parameters across statevector, exact Pauli propagation and PennyLane tensor devices. A central finite-difference check validates the full nonlinear loss gradient. MPS uses an untruncated small case here; this does not establish equivalence under aggressive truncation.

The separate IQP comparison uses a commuting X/XX circuit with Z correlations. Restricting the correlation axes changes the register capacity, so it is **not** a backend substitution for the default noncommuting PCE circuit. Exact VN enumeration is used only for small-system correctness; sampled VN gradients differentiate a stochastic empirical objective.

In [ ]:
validation_rows = []
small_problem = QUBO.maxcut(3, [(0, 1, 1.0), (1, 2, 2.0), (0, 2, 3.0)])

def finite_difference(model, angles):
    eps = 1e-6
    eye = np.eye(len(angles)) * eps
    return np.array([(model.loss(angles + e) - model.loss(angles - e))/(2*eps) for e in eye])

def check_group(group, common, variants):
    reference = PCE(small_problem, backend="torch", device="default.qubit", diff_method="backprop", **common)
    angles = np.linspace(-0.53, 0.71, reference.n_parameters)
    expected_value = reference.loss(angles)
    expected_gradient = finite_difference(reference, angles)
    for label, execution in variants:
        model = PCE(small_problem, backend="torch", **common, **execution)
        start = perf_counter(); value = model.loss(angles); gradient = model.gradient(angles)
        elapsed = perf_counter() - start
        error = np.max(np.abs(gradient - expected_gradient))
        assert abs(value - expected_value) < 1e-9
        assert error < 3e-6
        validation_rows.append({"circuit": group, "simulator": label, "loss_error": abs(value - expected_value),
                                "max_gradient_error": error, "cold_forward_and_gradient_seconds": elapsed})

variants = [
    ("default.qubit", {"device": "default.qubit", "diff_method": "backprop"}),
    ("lightning.qubit", {"device": "lightning.qubit", "diff_method": "adjoint"}),
    ("Pauli propagation · exact", {"device": "PauliPropagationSimulator", "diff_method": "backprop",
                                  "device_kwargs": {"k": None, "prune": False}}),
]
try:
    import quimb
except ImportError:
    print("Optional MPS/TN checks skipped: install quimb and cotengra.")
else:
    variants += [(f"PennyLane {method.upper()}", {"device": "default.tensor", "diff_method": "parameter-shift",
                                                "device_kwargs": {"method": method}}) for method in ["mps", "tn"]]
check_group("Y/Z/ZZ · XYZ readout", {}, variants)
iqp_common = {"n_qubits": 3, "paulis": "Z", "circuit": {
    "ansatz": "AngleAnsatz", "ansatz_config": {"n_layers": 2,
        "rotations": ["X 0", "X 1", "X 2", "XX 0 1", "XX 1 2"], "include_imprimitive": False}}}
check_group("Commuting X/XX · Z readout", iqp_common, [
    (f"IQP {backend}", {"device": "IQPSimulator", "diff_method": "backprop",
                       "device_kwargs": {"backend": backend, "vn_sampling": "enumerate"}})
    for backend in ["tn", "van_den_nest", "auto"]])
validation = pd.DataFrame(validation_rows)
display(validation)

## 5. Draw the trained models

These are the first graph's trained circuits. Matplotlib is the default graphical representation. `output="tikz", return_tex=False` displays a high-quality inline preview and cleans up temporary rendering files. To retain editable source **in memory**, use `tex = draw_pce(..., output="tikz")`. Saving a file is a separate explicit action. Slash-separated meter labels (for example X/Y) indicate alternative settings on separate circuit preparations, not simultaneous incompatible measurements. `pce_model.pauli_labels` lists the individual observables.

In [ ]:
example_graph, example_seed = graph_specs[0]["name"], SEEDS[0]
pce_model, pce_result = runs[(example_graph, example_seed, "PCE")]
qaoa_model, qaoa_result = runs[(example_graph, example_seed, "QAOA")]
for model, result, draw in [(pce_model, pce_result, draw_pce), (qaoa_model, qaoa_result, draw_qaoa)]:
    fig, ax = draw(model, parameters=result, output="matplotlib")
    plt.show()
    draw(model, parameters=result, output="tikz", return_tex=False,
         tikz_options={"standalone": True, "symbolic_parameters": True, "compress": True})

## Interpretation

- Compare **cut weights**, not the two different training losses. Inspect raw results to see how much the final classical sweep contributes.
- Fewer PCE qubits do not automatically imply less total work: correlation count, ansatz parameters, simulator choice and optimization behavior matter. QAOA's probability readout here is exponential in its register size and is used only for these small examples.
- Three graphs and three seeds are an instructional experiment, not statistical evidence of general superiority, a reproduction of the paper's benchmark, or a quantum advantage.
- The correctness checks are analytic and small. Noise, finite shots, Monte Carlo VN uncertainty, tensor truncation and approximate propagation need separate budgets and diagnostics.
- To extend the study, vary graph families, correlation order, PCE depth, QAOA depth, restart budgets and training steps; report quantum-only and postprocessed solutions separately.